<a href="https://colab.research.google.com/github/shin584/project/blob/3D_simulation/Feature_Stacking_Test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from transformers import AutoTokenizer, AutoModel
from google.colab import drive

# 1. 구글 드라이브 마운트
drive.mount('/content/drive')

# 2. 저장된 드라이브 경로 지정
model_path = '/content/drive/MyDrive/project_shared/models/NT_cas12a_fintuned_model'

# 3. 토크나이저 및 모델 로드
# 특징 추출기로 사용하기 위해 output_hidden_states=True 파라미터를 추가합니다.
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModel.from_pretrained(model_path, output_hidden_states=True)

print("파인튜닝된 NT 모델(Feature Extractor) 로드 완료!")

Mounted at /content/drive


Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: /content/drive/MyDrive/project_shared/models/NT_cas12a_fintuned_model
Key                        | Status     | 
---------------------------+------------+-
classifier.dense.bias      | UNEXPECTED | 
classifier.dense.weight    | UNEXPECTED | 
classifier.out_proj.bias   | UNEXPECTED | 
classifier.out_proj.weight | UNEXPECTED | 
pooler.dense.bias          | MISSING    | 
pooler.dense.weight        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


파인튜닝된 NT 모델(Feature Extractor) 로드 완료!


In [2]:
import torch
import numpy as np

# GPU 사용 가능 여부 확인 및 모델 이동
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval() # 평가(추론) 모드로 전환 (드롭아웃 등 비활성화)

def get_flattened_embeddings(sequences, tokenizer, model, batch_size=32):
    """
    DNA 서열 리스트를 입력받아 Flattening된 임베딩 텐서를 반환하는 함수
    """
    all_embeddings = []

    # 데이터를 배치(Batch) 단위로 쪼개서 처리 (메모리 초과 방지)
    for i in range(0, len(sequences), batch_size):
        batch_seqs = sequences[i : i + batch_size]

        # 1. 토큰화 (길이를 맞추기 위해 패딩 적용)
        inputs = tokenizer(
            batch_seqs,
            padding=True,       # 배치 내 최대 길이에 맞춰 빈칸(패딩) 채움
            truncation=True,    # 모델 최대 허용 길이 초과 시 자름
            return_tensors="pt" # PyTorch 텐서 형태로 반환
        ).to(device)

        # 2. 모델 추론 (기울기 계산 비활성화로 메모리 절약)
        with torch.no_grad():
            outputs = model(**inputs)

            # [batch_size, sequence_length, hidden_size] 형태의 3차원 텐서 추출
            last_hidden_state = outputs.last_hidden_state

            # 3. 평탄화(Flattening) 핵심 작업
            # .view(배치크기, -1)을 사용하면 나머지 차원(seq_len * hidden_size)이 1차원으로 길게 펴집니다.
            flattened_batch = last_hidden_state.view(last_hidden_state.size(0), -1)

            # 4. CPU 메모리로 옮긴 후 Numpy 배열로 변환하여 저장
            all_embeddings.append(flattened_batch.cpu().numpy())

    # 리스트에 담긴 배치별 결과를 하나의 거대한 Numpy 2차원 배열로 병합
    return np.vstack(all_embeddings)

# --- [테스트 실행] ---
# 가상의 CRISPR 타겟 서열 2개를 예시로 테스트합니다.
sample_sequences = [
    "ATCGATCGATCGATCGATCGATCGAT",
    "GCTAGCTAGCTAGCTAGCTAGCTAGC"
]

print("임베딩 추출 및 Flattening 진행 중...")
flattened_features = get_flattened_embeddings(sample_sequences, tokenizer, model, batch_size=2)

print("\n[변환 결과 확인]")
print(f"변환된 데이터의 형태(Shape): {flattened_features.shape}")
# 출력 예시: (2, 20736)
# -> 2개의 서열이 각각 20,736개의 숫자(특성)로 일렬로 완벽하게 펴졌음을 의미합니다.

임베딩 추출 및 Flattening 진행 중...

[변환 결과 확인]
변환된 데이터의 형태(Shape): (2, 8960)


In [7]:
!pip install viennarna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.4/13.4 MB 48.7 MB/s eta 0:00:00


In [9]:
!pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 36.3 MB/s eta 0:00:00


In [10]:
import math
import numpy as np
import RNA
from Bio.Seq import Seq
from Bio.SeqUtils import gc_fraction
from Bio.SeqUtils import MeltingTemp as mt

def extract_thermodynamic_features(sequences):
    """
    서열 리스트를 입력받아 4가지 물리/열역학적 스칼라 피처를 계산하여 반환합니다.
    (개선점 1, 2, 3 모두 반영 완료)
    """
    features = []

    for seq in sequences:
        # ---------------------------------------------------------
        # [개선점 2] 1. 최소 자유 에너지 (MFE) 계산 - ViennaRNA
        # DNA 서열을 RNA 서열로 명시적 변환 (T -> U) 후 계산
        # ---------------------------------------------------------
        rna_seq = seq.replace("T", "U")
        mfe_structure, mfe_value = RNA.fold(rna_seq)

        # ---------------------------------------------------------
        # [개선점 1] 2. DNA-RNA 하이브리드 결합 에너지 (ΔG) - RNAduplex
        # Target DNA로부터 상보 Guide RNA를 생성하여 이종 결합 안정성 평가
        # ---------------------------------------------------------
        # 타겟 DNA의 역상보 서열을 구하고 RNA로 전사 (예: ATCG -> CGAU)
        guide_rna = str(Seq(seq).reverse_complement().transcribe())
        # ViennaRNA 파라미터 적용을 위해 Target DNA도 임시 RNA(U)로 치환
        target_rna_proxy = seq.replace("T", "U")

        # duplexfold 수행을 통해 이종 결합 구조 에너지 추출
        duplex_result = RNA.duplexfold(guide_rna, target_rna_proxy)
        dg_value = duplex_result.energy

        # ---------------------------------------------------------
        # 3. 녹는점 (Tm) 계산 - Biopython
        # ---------------------------------------------------------
        tm_value = mt.Tm_NN(seq, nn_table=mt.DNA_NN4)

        # ---------------------------------------------------------
        # 4. GC 함량 (GC Content) 계산 - Biopython
        # ---------------------------------------------------------
        gc_value = gc_fraction(seq) * 100.0

        # ---------------------------------------------------------
        # [개선점 3] 결측치(NaN) 예외 처리
        # 수학적 계산 중 nan이 발생할 경우 구조/결합이 없는 것으로 간주(0.0)
        # ---------------------------------------------------------
        if math.isnan(mfe_value): mfe_value = 0.0
        if math.isnan(dg_value): dg_value = 0.0
        if math.isnan(tm_value): tm_value = 0.0
        if math.isnan(gc_value): gc_value = 0.0

        features.append([mfe_value, dg_value, tm_value, gc_value])

    return np.array(features)

print("물리·열역학적 스칼라 피처 연산 중 (모든 개선안 반영)...")
# 2단계에서 생성된 sample_sequences와 flattened_features를 그대로 활용합니다.
scalar_features = extract_thermodynamic_features(sample_sequences)

print(f"추출된 스칼라 피처 형태(Shape): {scalar_features.shape}")
# 기대 출력: (데이터 개수, 4)

print("\nNT 임베딩과 열역학 피처를 병합합니다...")
# np.hstack을 사용하여 트랜스포머 임베딩과 스칼라 수치를 가로로 이어 붙입니다.
final_X_train = np.hstack((flattened_features, scalar_features))

print("\n[최종 완성된 융합 데이터 세트 확인]")
print(f"융합된 최종 데이터의 형태(Shape): {final_X_train.shape}")
# 기대 출력: (데이터 개수, 8964)

물리·열역학적 스칼라 피처 연산 중 (모든 개선안 반영)...
추출된 스칼라 피처 형태(Shape): (2, 4)

NT 임베딩과 열역학 피처를 병합합니다...

[최종 완성된 융합 데이터 세트 확인]
융합된 최종 데이터의 형태(Shape): (2, 8964)


In [12]:
import pandas as pd
import numpy as np
import xgboost as xgb
from scipy.stats import spearmanr, pearsonr
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ---------------------------------------------------------
# 1. 실제 CSV 데이터 로드 및 분리
# ---------------------------------------------------------
print("[데이터 준비] CSV 파일을 로드합니다...")
train_df = pd.read_csv("train.csv")
val_df = pd.read_csv("val.csv")
test_df = pd.read_csv("test.csv")

# 서열(X)과 절단 효율 정답(Y) 추출
train_seqs = train_df['input_sequence'].tolist()
y_train = train_df['score'].values

val_seqs = val_df['input_sequence'].tolist()
y_val = val_df['score'].values

test_seqs = test_df['input_sequence'].tolist()
y_test = test_df['score'].values

# ---------------------------------------------------------
# 2. 파운데이션 모델 임베딩 및 열역학 피처 추출 (시간 소요됨)
# ---------------------------------------------------------
print("\nTrain 데이터 특징 추출 중...")
X_train_emb = get_flattened_embeddings(train_seqs, tokenizer, model, batch_size=32)
X_train_thermo = extract_thermodynamic_features(train_seqs)
X_train = np.hstack((X_train_emb, X_train_thermo))

print("Validation 데이터 특징 추출 중...")
X_val_emb = get_flattened_embeddings(val_seqs, tokenizer, model, batch_size=32)
X_val_thermo = extract_thermodynamic_features(val_seqs)
X_val = np.hstack((X_val_emb, X_val_thermo))

print("Test 데이터 특징 추출 중...")
X_test_emb = get_flattened_embeddings(test_seqs, tokenizer, model, batch_size=32)
X_test_thermo = extract_thermodynamic_features(test_seqs)
X_test = np.hstack((X_test_emb, X_test_thermo))

# ---------------------------------------------------------
# 3. XGBoost 융합 모델 학습 및 검증
# ---------------------------------------------------------
print("\n[Phase 4] XGBoost 모델 학습 시작...")

# 모델 초기화
xgb_model = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    tree_method='hist',
    early_stopping_rounds=20
)

# 학습 (Validation 세트로 Early Stopping 적용)
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)
print("XGBoost 모델 학습 및 검증 완료.")

# ---------------------------------------------------------
# 4. 최종 평가 (독립된 Test 세트)
# ---------------------------------------------------------
print("\n[최종 평가] 독립된 Test 데이터셋에 대한 추론을 진행합니다.")
predictions = xgb_model.predict(X_test)

if np.std(predictions) == 0 or np.std(y_test) == 0:
    test_rho, test_pearson = 0.0, 0.0
else:
    test_rho, _ = spearmanr(y_test, predictions)
    test_pearson, _ = pearsonr(y_test, predictions)

test_mae = mean_absolute_error(y_test, predictions)
test_mse = mean_squared_error(y_test, predictions)

print("\n[Phase 4-2 이종 모델 융합 최종 성능 지표]")
print("-" * 40)
print(f"Spearman correlation (ρ) : {test_rho:.4f} (비교 핵심 지표)")
print(f"Pearson correlation (r)  : {test_pearson:.4f}")
print(f"Mean Absolute Error (MAE): {test_mae:.4f}")
print(f"Mean Squared Error (MSE) : {test_mse:.4f}")
print("-" * 40)

[데이터 준비] CSV 파일을 로드합니다...

Train 데이터 특징 추출 중...
Validation 데이터 특징 추출 중...
Test 데이터 특징 추출 중...

[Phase 4] XGBoost 모델 학습 시작...
XGBoost 모델 학습 및 검증 완료.

[최종 평가] 독립된 Test 데이터셋에 대한 추론을 진행합니다.

[Phase 4-2 이종 모델 융합 최종 성능 지표]
----------------------------------------
Spearman correlation (ρ) : 0.7981 (비교 핵심 지표)
Pearson correlation (r)  : 0.8090
Mean Absolute Error (MAE): 0.1088
Mean Squared Error (MSE) : 0.0196
----------------------------------------
